# VLM Hallucination Probe
Collects hidden states from a VLM on POPE, balances the data, trains a probe.

**Run all cells top to bottom. Only edit the CONFIG cell.**

In [ ]:
# ── Check GPU ────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Runtime > Change runtime type > T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install -q transformers>=4.49.0 accelerate qwen-vl-utils torchvision scikit-learn pandas tqdm huggingface_hub

In [ ]:
# ── Mount Google Drive (recommended — saves results across sessions) ──
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE_DIR = '/content/drive/MyDrive/189G-probe-results'
import os; os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Drive mounted. Results will be saved to:', DRIVE_SAVE_DIR)

In [ ]:
# ── Download POPE dataset ─────────────────────────────────────
import os
from huggingface_hub import hf_hub_download

os.makedirs('POPE/Full', exist_ok=True)
for split in ['adversarial', 'popular', 'random']:
    dst = f'POPE/Full/{split}-00000-of-00001.parquet'
    if os.path.exists(dst):
        print(f'  {split}: already downloaded')
        continue
    print(f'  Downloading {split}...')
    hf_hub_download(
        repo_id='lmms-lab/POPE', repo_type='dataset',
        filename=f'Full/{split}-00000-of-00001.parquet',
        local_dir='POPE',
    )
print('POPE data ready.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                        CONFIG                               ║
# ║            Edit this cell, then run all below               ║
# ╚══════════════════════════════════════════════════════════════╝

# Model to run. Options:
#   'Qwen/Qwen2.5-VL-3B-Instruct'           ~8 GB VRAM  (matches Gege — recommended)
#   'OpenGVLab/InternVL2-2B'                ~5 GB VRAM
#   'microsoft/Phi-3.5-vision-instruct'     ~8 GB VRAM
#   'deepseek-ai/deepseek-vl-1.3b-base'    ~4 GB VRAM  (matches Gege)
#   'openbmb/MiniCPM-V-2'                   ~8 GB VRAM
MODEL_NAME   = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Number of POPE samples to process.
# 500 takes ~15 min on T4 and gives a usable probe.
# Set to None to run all 9000 (takes ~3-4 hrs).
MAX_SAMPLES  = 500

# POPE splits to use: 'adversarial', 'popular', 'random', or 'all'
POPE_SPLIT   = 'all'

# Number of text few-shot examples prepended to each prompt
FEW_SHOT_N   = 3

# Hidden state pooling: 'last' (last output token) or 'mean' (mean over output tokens)
POOLING      = 'last'

# ── Derived paths (don't edit) ────────────────────────────────
model_short  = MODEL_NAME.replace('/', '_')
NPZ_RAW      = f'hidden_states_{model_short}_{POPE_SPLIT}_{POOLING}.npz'
NPZ_BALANCED = f'balanced_{model_short}_{POPE_SPLIT}_{POOLING}.npz'
TEST_IDS_FILE = 'test_ids.json'
print(f'Model:        {MODEL_NAME}')
print(f'Max samples:  {MAX_SAMPLES}')
print(f'Output file:  {NPZ_RAW}')

In [ ]:
# ── Load model ────────────────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoTokenizer

DTYPE  = torch.bfloat16
DEVICE = 'cuda'

def detect_family(name):
    n = name.lower()
    if 'qwen2.5-vl' in n or 'qwen2-vl' in n: return 'qwen2vl'
    if 'phi-3.5-vision' in n or 'phi-3-vision' in n: return 'phi3vision'
    if 'internvl' in n: return 'internvl'
    if 'deepseek-vl' in n: return 'deepseek'
    if 'minicpm' in n: return 'minicpm'
    return 'generic'

FAMILY = detect_family(MODEL_NAME)
print(f'Model family: {FAMILY}')

if FAMILY == 'qwen2vl':
    from transformers import Qwen2_5_VLForConditionalGeneration
    model     = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto')
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    tokenizer = processor.tokenizer

elif FAMILY == 'phi3vision':
    from transformers import AutoModelForCausalLM
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto',
                    trust_remote_code=True, _attn_implementation='eager')
    processor = AutoProcessor.from_pretrained(
                    MODEL_NAME, trust_remote_code=True, num_crops=4)
    tokenizer = processor.tokenizer

elif FAMILY == 'internvl':
    from transformers import AutoModel
    model     = AutoModel.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto',
                    trust_remote_code=True, low_cpu_mem_usage=True)
    tokenizer = AutoTokenizer.from_pretrained(
                    MODEL_NAME, trust_remote_code=True, use_fast=False)
    processor = None

elif FAMILY == 'deepseek':
    from transformers import AutoModelForCausalLM
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto',
                    trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer = processor.tokenizer

elif FAMILY == 'minicpm':
    from transformers import AutoModel
    model     = AutoModel.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto',
                    trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    processor = None

else:
    from transformers import AutoModelForCausalLM
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME, torch_dtype=DTYPE, device_map='auto',
                    trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer = processor.tokenizer

model.eval()
print('Model loaded.')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ── Load POPE data & set up test split ───────────────────────
import json
import numpy as np
import pandas as pd
from io import BytesIO
from PIL import Image

splits_to_load = (['adversarial','popular','random'] if POPE_SPLIT == 'all'
                  else [POPE_SPLIT])
dfs = []
for s in splits_to_load:
    df = pd.read_parquet(f'POPE/Full/{s}-00000-of-00001.parquet')
    df['pope_split'] = s
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all)} samples.')
print(df_all['answer'].value_counts().to_string())

if MAX_SAMPLES is not None:
    df_all = df_all.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
    print(f'Subsampled to {len(df_all)} samples.')

# Shared test split ─────────────────────────────────────────
# If test_ids.json exists (from a previous model run), reuse it.
# Otherwise create it so all models share the same test questions.
if os.path.exists(TEST_IDS_FILE):
    with open(TEST_IDS_FILE) as f:
        test_ids = set(json.load(f))
    print(f'Loaded {len(test_ids)} shared test IDs.')
else:
    rng   = np.random.default_rng(42)
    n_per = int(len(df_all) * 0.2) // 2
    yes_q = df_all[df_all['answer']=='yes']['question_id'].astype(str).tolist()
    no_q  = df_all[df_all['answer']=='no' ]['question_id'].astype(str).tolist()
    test_ids = set(
        rng.choice(yes_q, size=min(n_per,len(yes_q)), replace=False).tolist() +
        rng.choice(no_q,  size=min(n_per,len(no_q)),  replace=False).tolist()
    )
    with open(TEST_IDS_FILE, 'w') as f:
        json.dump(list(test_ids), f)
    print(f'Created shared test split: {len(test_ids)} IDs.')

In [ ]:
# ── Helper functions ──────────────────────────────────────────
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

FEW_SHOT_EXAMPLES = [
    ('Is there a dog in the image?',       'No'),
    ('Is there a chair in the image?',     'Yes'),
    ('Is there an airplane in the image?', 'No'),
    ('Is there a cup in the image?',       'Yes'),
    ('Is there a fire hydrant in the image?', 'No'),
][:FEW_SHOT_N]

def get_image(img_field):
    if hasattr(img_field, 'convert'):  return img_field.convert('RGB')
    if isinstance(img_field, dict):
        if img_field.get('bytes'):     return Image.open(BytesIO(img_field['bytes'])).convert('RGB')
        if img_field.get('path'):      return Image.open(img_field['path']).convert('RGB')
    if isinstance(img_field, bytes):   return Image.open(BytesIO(img_field)).convert('RGB')
    return None

def exact_match(pred, gold):
    p = pred.strip().lower().rstrip('.,!?')
    g = gold.strip().lower().rstrip('.,!?')
    return p == g or p.startswith(g)

def build_prompt(question):
    lines = ['Here are some example questions and answers about images:']
    for q, a in FEW_SHOT_EXAMPLES:
        lines.append(f'Q: {q}  A: {a}.')
    lines += ['', "Now answer the following question. Reply with only 'yes' or 'no'.",
              f'Q: {question}']
    return '\n'.join(lines)

# InternVL image transform
_internvl_transform = T.Compose([
    T.Lambda(lambda img: img.convert('RGB')),
    T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])

def encode(image, question, include_answer=None):
    """Encode image+question (and optionally answer) into model input tensors."""
    prompt = build_prompt(question)

    if FAMILY == 'qwen2vl':
        msgs = [{'role':'user','content':[
            {'type':'image','image':image},{'type':'text','text':prompt}]}]
        if include_answer:
            msgs.append({'role':'assistant','content':include_answer})
        chat = processor.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=(include_answer is None))
        inp  = processor(text=[chat], images=[image], padding=True, return_tensors='pt')

    elif FAMILY == 'phi3vision':
        answer_part = include_answer if include_answer else ''
        text = (f'<|user|>\n<|image_1|>\n{prompt}<|end|>\n'
                f'<|assistant|>{answer_part}' + ('<|end|>' if include_answer else ''))
        inp  = processor(text=text, images=[image], return_tensors='pt')

    elif FAMILY == 'internvl':
        pv   = _internvl_transform(image).unsqueeze(0).to(DTYPE).to(model.device)
        q    = f'<image>\n{prompt}'
        if include_answer:
            q += f'\n<|im_start|>assistant\n{include_answer}<|im_end|>'
        ids  = tokenizer(q, return_tensors='pt')
        inp  = {'pixel_values': pv,
                'input_ids': ids['input_ids'].to(model.device),
                'attention_mask': ids['attention_mask'].to(model.device)}
        return inp

    elif FAMILY == 'deepseek':
        msgs = [{'role':'user','content':[{'type':'image'},{'type':'text','text':prompt}]}]
        if include_answer:
            msgs.append({'role':'assistant','content':include_answer})
        chat = processor.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=(include_answer is None))
        inp  = processor(text=[chat], images=[image], return_tensors='pt')

    else:  # minicpm / generic fallback
        msgs = [{'role':'user','content':[{'type':'image','image':image},
                                          {'type':'text','text':prompt}]}]
        if include_answer:
            msgs.append({'role':'assistant','content':include_answer})
        chat = processor.apply_chat_template(msgs, tokenize=False,
                   add_generation_prompt=(include_answer is None))
        inp  = processor(text=[chat], images=[image], padding=True, return_tensors='pt')

    return {k: v.to(model.device) for k, v in inp.items()}

print('Helpers defined.')

In [ ]:
# ── Collect hidden states ─────────────────────────────────────
from tqdm.notebook import tqdm

all_features, all_labels, all_preds, all_gts, all_ids, all_splits = [], [], [], [], [], []

for _, row in tqdm(df_all.iterrows(), total=len(df_all)):
    qid      = str(row['question_id'])
    question = row['question']
    gt       = row['answer'].strip().lower()
    split_tag = 'test' if qid in test_ids else 'train_val'

    image = get_image(row['image'])
    if image is None:
        continue

    try:
        # Step 1: encode prompt & generate answer
        prompt_inputs = encode(image, question)
        prompt_len    = prompt_inputs['input_ids'].shape[1]
        gen_kwargs    = {k: v for k, v in prompt_inputs.items() if isinstance(v, torch.Tensor)}

        with torch.no_grad():
            gen_ids  = model.generate(**gen_kwargs, max_new_tokens=5,
                                      do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
        gen_text = tokenizer.decode(gen_ids[0, prompt_len:],
                                    skip_special_tokens=True).strip().lower()

        # Step 2: EM label
        label = 0 if exact_match(gen_text, gt) else 1  # 0=factual, 1=nonfactual

        # Step 3: teacher-forcing forward pass on (prompt + answer)
        full_inputs = encode(image, question, include_answer=gen_text)
        full_len    = full_inputs['input_ids'].shape[1]
        n_new       = max(full_len - prompt_len, 1)
        fwd_kwargs  = {k: v for k, v in full_inputs.items() if isinstance(v, torch.Tensor)}

        with torch.no_grad():
            out = model(**fwd_kwargs, output_hidden_states=True, return_dict=True)

        # Step 4: extract hidden state at output token position
        features = []
        for h in out.hidden_states:
            h_seq = h[0]
            vec   = (h_seq[-1] if POOLING == 'last'
                     else h_seq[-n_new:].mean(dim=0))
            features.append(vec.float().cpu().numpy())

    except Exception as e:
        print(f'Skip {qid}: {e}')
        continue

    all_features.append(np.stack(features))
    all_labels.append(label)
    all_preds.append(gen_text)
    all_gts.append(gt)
    all_ids.append(qid)
    all_splits.append(split_tag)

all_features = np.stack(all_features)
all_labels   = np.array(all_labels)
split_arr    = np.array(all_splits)

print(f'\nFeatures: {all_features.shape}  (samples x layers x hidden_dim)')
uniq, cnts = np.unique(all_labels, return_counts=True)
for u, c in zip(uniq, cnts):
    name = 'factual' if u == 0 else 'nonfactual'
    print(f'  label={u} ({name}): {c}  ({100*c/len(all_labels):.1f}%)')
print(f'  model EM accuracy: {100*(all_labels==0).mean():.1f}%')

np.savez(NPZ_RAW, features=all_features, labels=all_labels,
         preds=np.array(all_preds), gts=np.array(all_gts),
         ids=np.array(all_ids), splits=split_arr)
print(f'Saved raw → {NPZ_RAW}')

In [ ]:
# ── Balance dataset → 50/50 factual/nonfactual ────────────────
import os

os.makedirs('balanced', exist_ok=True)

tv_mask   = split_arr == 'train_val'
test_mask = split_arr == 'test'

tv_labels    = all_labels[tv_mask]
n_factual    = (tv_labels == 0).sum()
n_nonfactual = (tv_labels == 1).sum()
n_min        = min(n_factual, n_nonfactual)
majority     = 'factual' if n_factual > n_nonfactual else 'nonfactual'

print(f'train_val — factual: {n_factual}, nonfactual: {n_nonfactual}')
print(f'Majority class: {majority}')
print(f'Subsample ratio: {n_nonfactual}/{n_factual} = {n_nonfactual/max(n_factual,1):.4f}  (fraction of factual to keep)')
print(f'Keeping {n_min} per class = {2*n_min} train_val samples total')

rng     = np.random.default_rng(42)
tv_idx  = np.where(tv_mask)[0]
pos_idx = tv_idx[all_labels[tv_idx] == 1]
neg_idx = tv_idx[all_labels[tv_idx] == 0]
sel     = np.concatenate([rng.choice(pos_idx, n_min, replace=False),
                          rng.choice(neg_idx, n_min, replace=False)])
rng.shuffle(sel)

keep = np.sort(np.concatenate([sel, np.where(test_mask)[0]]))

np.savez(f'balanced/{NPZ_RAW}',
         features = all_features[keep],
         labels   = all_labels[keep],
         preds    = np.array(all_preds)[keep],
         gts      = np.array(all_gts)[keep],
         ids      = np.array(all_ids)[keep],
         splits   = split_arr[keep])
print(f'Saved balanced → balanced/{NPZ_RAW}')

In [ ]:
# ── Train probe ───────────────────────────────────────────────
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

PROBE_DEVICE = 'cuda'
SWEEP_LAYERS = True   # set False to train only on LAYER_IDX
LAYER_IDX    = 20
BATCH_SIZE   = 64
EPOCHS       = 50
LR           = 1e-4
PATIENCE     = 5
HIDDEN       = 512

# Load balanced data
d      = np.load(f'balanced/{NPZ_RAW}', allow_pickle=True)
feats  = d['features']   # [N, L, D]
labels = d['labels']
splits = d['splits']
n_layers, hidden_dim = feats.shape[1], feats.shape[2]
print(f'Loaded: {feats.shape[0]} samples | {n_layers} layers | dim={hidden_dim}')

def balance_split(feats, labels, splits):
    tv = splits == 'train_val'
    X_tv, y_tv = feats[tv], labels[tv]
    X_te, y_te = feats[~tv], labels[~tv]
    n_val = max(1, int(len(X_tv) * 0.2))
    return X_tv[n_val:], y_tv[n_val:], X_tv[:n_val], y_tv[:n_val], X_te, y_te

def train_layer(layer_idx):
    X_tr, y_tr, X_v, y_v, X_te, y_te = balance_split(
        feats[:, layer_idx, :], labels, splits)

    def t(X, y):
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))
    Xt, yt = t(X_tr, y_tr)
    Xv, yv = t(X_v,  y_v)
    Xte    = torch.tensor(X_te, dtype=torch.float32)

    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=True)

    probe = nn.Sequential(
        nn.Linear(hidden_dim, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 1)
    ).to(PROBE_DEVICE)
    opt  = torch.optim.Adam(probe.parameters(), lr=LR)
    crit = nn.BCEWithLogitsLoss()

    best_val, best_state, patience_ctr = -1, None, 0
    for epoch in range(1, EPOCHS + 1):
        probe.train()
        for bx, by in loader:
            bx, by = bx.to(PROBE_DEVICE), by.to(PROBE_DEVICE)
            loss = crit(probe(bx).squeeze(-1), by)
            opt.zero_grad(); loss.backward(); opt.step()
        probe.eval()
        with torch.no_grad():
            vp = (torch.sigmoid(probe(Xv.to(PROBE_DEVICE)).squeeze(-1)) > 0.5).long().cpu().numpy()
        val_acc = accuracy_score(y_v, vp)
        if val_acc > best_val:
            best_val  = val_acc
            best_state = {k: v.cpu().clone() for k, v in probe.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    probe.load_state_dict(best_state)
    probe.eval()
    with torch.no_grad():
        tp = (torch.sigmoid(probe(Xte.to(PROBE_DEVICE)).squeeze(-1)) > 0.5).long().cpu().numpy()
    return best_val, accuracy_score(y_te, tp), f1_score(y_te, tp, zero_division=0)

if SWEEP_LAYERS:
    print(f'Sweeping {n_layers} layers...')
    results = []
    for li in range(n_layers):
        val_acc, test_acc, test_f1 = train_layer(li)
        results.append((li, val_acc, test_acc, test_f1))
        print(f'  Layer {li:3d} | val={val_acc:.4f} | test_acc={test_acc:.4f} | test_f1={test_f1:.4f}')
    best = max(results, key=lambda x: x[1])
    print(f'\nBest layer: {best[0]}  val={best[1]:.4f}  test_acc={best[2]:.4f}  test_f1={best[3]:.4f}')
else:
    val_acc, test_acc, test_f1 = train_layer(LAYER_IDX)
    print(f'Layer {LAYER_IDX}: val={val_acc:.4f} | test_acc={test_acc:.4f} | test_f1={test_f1:.4f}')

In [ ]:
# ── Save results to Google Drive ──────────────────────────────
import shutil

# Raw hidden states
shutil.copy(NPZ_RAW, f'{DRIVE_SAVE_DIR}/{NPZ_RAW}')
# Balanced
shutil.copy(f'balanced/{NPZ_RAW}', f'{DRIVE_SAVE_DIR}/balanced_{NPZ_RAW}')
# Shared test split
shutil.copy(TEST_IDS_FILE, f'{DRIVE_SAVE_DIR}/{TEST_IDS_FILE}')

print('Saved to Drive:')
print(f'  {DRIVE_SAVE_DIR}/{NPZ_RAW}')
print(f'  {DRIVE_SAVE_DIR}/balanced_{NPZ_RAW}')
print(f'  {DRIVE_SAVE_DIR}/{TEST_IDS_FILE}')